# W12 · Lab A · Exact Inference and Differentiable Gaussian Draws

Advanced Machine Learning for Artificial Intelligence · November 17, 2026

Run from the top on CPU. The cells use NumPy and Matplotlib. Notes S02–S04 and Examples 1–3 follow the same order. For each edit, write a prediction first and retain both outputs.


In [ ]:
RELEASE = 'https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12'
EXPECTED_MODULES = {'advml_w12.py': '35c9e6b9083341d768976feb55c08ed5b4f57e4aa4108b452611dce04b213218'}


In [ ]:
import hashlib
from pathlib import Path
import sys
import tempfile
from urllib.request import urlopen

source_work = tempfile.TemporaryDirectory(prefix="advml-w12-")
source_cache = Path(source_work.name)
for name, expected in EXPECTED_MODULES.items():
    local_paths = [Path.cwd() / "src" / name]
    data = next((p.read_bytes().replace(b"\r\n", b"\n") for p in local_paths if p.is_file()), None)
    if data is None:
        with urlopen(f"{RELEASE}/src/{name}", timeout=30) as response:
            data = response.read().replace(b"\r\n", b"\n")
    assert hashlib.sha256(data).hexdigest() == expected, f"Source hash mismatch: {name}"
    (source_cache / name).write_bytes(data)
sys.path.insert(0, str(source_cache.resolve()))


## S02 — Generate, Marginalize, Infer

**A01 · Condition on an observed light.** Multiply row by row, add, then normalize. Change PRIOR to [0.75, 0.25] after predicting the two joint masses and posterior.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F02-masses.png" alt="Condition on an observed light" width="820">


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from advml_w12 import finite_posterior, finite_elbo

PRIOR = np.array([.5, .5])
LIKELIHOOD = np.array([.2, .8])

def compute_masses(prior, likelihood):
    joint = prior * likelihood
    evidence = joint.sum()
    posterior = joint / evidence
    return joint, evidence, posterior

joint, evidence, posterior = compute_masses(PRIOR, LIKELIHOOD)
print("joint:", joint, "evidence:", evidence, "posterior:", posterior)


In [ ]:
baseline = compute_masses(np.array([.5, .5]), LIKELIHOOD)
changed = compute_masses(np.array([.75, .25]), LIKELIHOOD)
np.testing.assert_allclose(baseline[0], [.1, .4])
np.testing.assert_allclose(baseline[2], [.2, .8])
np.testing.assert_allclose(changed[2], [3/7, 4/7])
for actual, reference in zip(changed, finite_posterior([.75,.25], [.2,.8])):
    np.testing.assert_allclose(actual, reference)


## S03 — Where Does the ELBO Gap Come From?

**A02 · Move the approximation.** Set MASS_ON_ONE to 0.8 after predicting which KL becomes zero. The chart holds the generative model and evidence fixed.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F03-elbo.png" alt="Move the approximation" width="820">


In [ ]:
MASS_ON_ONE = .5
model_prior = np.array([.5, .5])

def compute_terms(q, prior, likelihood):
    expected_loglik = np.sum(q * np.log(likelihood))
    prior_kl = np.sum(q * np.log(q / prior))
    return expected_loglik - prior_kl, prior_kl

q = np.array([1 - MASS_ON_ONE, MASS_ON_ONE])
elbo, prior_kl = compute_terms(q, model_prior, LIKELIHOOD)
reference = finite_elbo(model_prior, LIKELIHOOD, q)
print(f"ELBO {elbo:.6f} | log evidence {reference['log_evidence']:.6f}")
print(f"Prior KL {prior_kl:.6f} | posterior KL {reference['posterior_kl']:.6f}")
weights = np.linspace(.005, .995, 100)
bounds = [compute_terms(np.array([1-w,w]), model_prior, LIKELIHOOD)[0] for w in weights]
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(weights, bounds, label="ELBO")
ax.axhline(reference["log_evidence"], color="green", linestyle="--", label="log evidence")
ax.vlines(MASS_ON_ONE, elbo, reference["log_evidence"], color="darkorange", label="gap")
ax.set(xlabel="q(z=1)", ylabel="nats")
ax.legend()
plt.show()


In [ ]:
loose = finite_elbo([.5,.5],[.2,.8],[.5,.5])
tight = finite_elbo([.5,.5],[.2,.8],[.2,.8])
np.testing.assert_allclose(compute_terms(np.array([.5,.5]), model_prior, LIKELIHOOD),
                           [-.916290731874, 0], atol=1e-10)
np.testing.assert_allclose(loose["log_evidence"] - loose["elbo"], loose["posterior_kl"])
np.testing.assert_allclose(tight["posterior_kl"], 0, atol=1e-12)


## S04 — From the Encoder to a Differentiable Draw

**A03 · Trace a Gaussian sample.** Predict z for mu=1, sigma=0.5 and epsilon=-2. Correct the flagged line inside gaussian_draw. Reuse the same noise on both sides of the central difference.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F04-reparameterize.png" alt="Trace a Gaussian sample" width="820">


In [ ]:
def gaussian_draw(mu, logvar, eps):
    # TODO A03: Repair this line. What power turns variance into standard deviation?
    scale = np.exp(logvar)
    return mu + scale * eps

def central_difference(fn, at, delta=1e-5):
    return (fn(at + delta) - fn(at - delta)) / (2 * delta)

mu, logvar, eps = 1., np.log(.25), -2.
guess = gaussian_draw(mu, logvar, eps)
derivative = central_difference(lambda lv: gaussian_draw(mu, lv, eps), logvar)
print("z:", guess, "finite difference:", derivative)
print("Check after edit:", "matched" if np.allclose([guess, derivative], [0., -.5]) else "edit the scale line")


## S04 — From the Encoder to a Differentiable Draw

**A04 · Check fixed-noise gradients and one-coordinate KL.** Swap epsilon from -2 to +2. Anticipate the derivative sign. Write one sentence explaining why the fixed draw matters.


In [ ]:
def analytic_grad(logvar, eps):
    return .5 * np.exp(.5 * logvar) * eps

def gaussian_kl(mu, logvar):
    return .5 * (mu**2 + np.exp(logvar) - 1 - logvar)

np.testing.assert_allclose([analytic_grad(np.log(.25), e) for e in (-2.,2.)], [-.5,.5])
np.testing.assert_allclose(gaussian_kl(1., np.log(.25)), .8181471805599453)
print("KL in nats:", gaussian_kl(1., np.log(.25)))
